In [ ]:


from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist, squareform
import umap
import seaborn as sns
def load_token_id_map(path: Path, encoding="utf-8"):
    """
    Legge file con due colonne: token <whitespace> id
    Ritorna (token_to_id, id_to_token)
    Parsing robusto: prende l'ultimo campo come id (split da destra).
    """
    token_to_id = {}
    id_to_token = {}

    with path.open("r", encoding=encoding) as f:
        for ln, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            # split da destra: ultimo "campo" deve essere l'id
            try:
                token, id_str = line.rsplit(None, 1)  # separa su whitespace, solo 1 split
            except ValueError:
                raise ValueError(f"Linea {ln} non ha 2 colonne: {line!r}")

            try:
                tid = int(id_str)
            except ValueError:
                raise ValueError(f"Linea {ln} id non intero: {id_str!r}")

            token_to_id[token] = tid
            id_to_token[tid] = token

    return token_to_id, id_to_token


def load_grid_structure(path: Path, token_to_id: dict, encoding="utf-8"):
    """
    Legge griglia 4x4 (whitespace-separated) come nel tuo esempio.
    Ritorna:
      GRID: list[list[str]]
      WORD_TO_POS: dict[token] -> (r,c)
      WORD_TO_TID: dict[token] -> id
      ID_TO_WORD: dict[id] -> token
      ROW_LABELS: list[str]
    """
    rows = []
    with path.open("r", encoding=encoding) as f:
        for ln, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            row = line.split()
            rows.append(row)

    if len(rows) != 4 or any(len(r) != 4 for r in rows):
        raise ValueError(f"Griglia attesa 4x4, trovata {len(rows)} righe con lunghezze {[len(r) for r in rows]}")

    GRID = rows

    WORD_TO_POS = {w: (r, c) for r, row in enumerate(GRID) for c, w in enumerate(row)}

    # Map token->id usando il vocabolario caricato
    missing = [w for w in WORD_TO_POS.keys() if w not in token_to_id]
    if missing:
        raise KeyError(
            "Token della griglia mancanti nel file token-id. Esempi: "
            + ", ".join(missing[:8])
            + (" ..." if len(missing) > 8 else "")
        )

    WORD_TO_TID = {w: token_to_id[w] for w in WORD_TO_POS.keys()}
    ID_TO_WORD = {tid: w for w, tid in WORD_TO_TID.items()}

    ROW_LABELS = [f"Row {r}: " + " ".join(GRID[r]) for r in range(4)]

    return GRID, WORD_TO_POS, WORD_TO_TID, ID_TO_WORD, ROW_LABELS


def load_embeddings_pt(pt_path: Path, emb_key="embeddings_last", ids_key="input_ids_last"):
    """
    Carica un .pt nel tuo formato e ritorna:
      ids_np: (L,) int64
      emb_np: (L, D) float32
    Gestisce anche il caso (1, L) e (1, L, D).
    """
    obj = torch.load(pt_path, map_location="cpu", weights_only=False)
    ids = obj[ids_key]
    emb = obj[emb_key]

    # squeeze batch dimension se presente
    if ids.ndim == 2 and ids.shape[0] == 1:
        ids = ids[0]
    if emb.ndim == 3 and emb.shape[0] == 1:
        emb = emb[0]

    ids_np = ids.detach().cpu().numpy()
    emb_np = emb.detach().cpu().float().numpy()

    return ids_np, emb_np

def grid_edges_from_grid(GRID, WORD_TO_TID, available_tids=None):
    """
    Ritorna lista di (tid1, tid2) per celle adiacenti (4-neighbors) in GRID.
    Se available_tids è dato, filtra solo coppie con entrambi in available_tids.
    """
    n_rows = len(GRID)
    n_cols = len(GRID[0]) if n_rows else 0

    edges = []
    for r in range(n_rows):
        for c in range(n_cols):
            w1 = GRID[r][c]
            t1 = WORD_TO_TID[w1]
            for rr, cc in [(r-1,c), (r+1,c), (r,c-1), (r,c+1)]:
                if 0 <= rr < n_rows and 0 <= cc < n_cols:
                    w2 = GRID[rr][cc]
                    t2 = WORD_TO_TID[w2]
                    if t1 < t2:
                        if available_tids is None or (t1 in available_tids and t2 in available_tids):
                            edges.append((t1, t2))
    return edges



def plot_window_pca_grid(
    all_ids_np,
    all_emb_np,
    GRID,
    WORD_TO_POS,
    WORD_TO_TID,
    ID_TO_WORD,
    row_colors,
    row_labels,
    NW=500,
    random_state=0,
    point_size=60,
    alpha=0.75,
    title_prefix="Single random walk",
):
    """
    - Prende ids/embeddings (numpy) di una sequenza
    - Finestra sugli ultimi NW token
    - PCA 2D sugli embeddings finestrati
    - Calcola centroidi per token-id
    - Disegna edges (griglia), scatter e annotazioni
    """
    L_total = len(all_ids_np)
    W = min(NW, L_total)

    win_ids = all_ids_np[-W:]
    win_emb = all_emb_np[-W:]

    print(f"Window: {len(win_ids)} timesteps (file length {L_total})")
    print(f"Unique tokens in window: {len(np.unique(win_ids))}")

    # PCA
    pca_win = PCA(n_components=2, random_state=random_state)
    Z_win = pca_win.fit_transform(win_emb)

    # Centroidi per token-id
    unique_win = np.unique(win_ids)
    centroids = {int(t): Z_win[win_ids == t].mean(axis=0) for t in unique_win}

    fig, ax = plt.subplots(figsize=(9, 7))

    # Edges della griglia (solo se entrambi i token appaiono in finestra)
    edges = grid_edges_from_grid(GRID, WORD_TO_TID, available_tids=set(centroids))
    for t1, t2 in edges:
        c1, c2 = centroids[t1], centroids[t2]
        ax.plot([c1[0], c2[0]], [c1[1], c2[1]], color="#cccccc", lw=1, zorder=1)

    legend_added = set()

    for tid in unique_win:
        tid_int = int(tid)
        mask = (win_ids == tid)

        # Se il token-id è nella griglia, ottengo la parola e la riga
        if tid_int in ID_TO_WORD:
            word = ID_TO_WORD[tid_int]
            row_g, _ = WORD_TO_POS[word]
            color = row_colors[row_g]
            label = row_labels[row_g] if row_g not in legend_added else None
            legend_added.add(row_g)
        else:
            # token non nella griglia
            word = str(tid_int)
            color = "grey"
            label = None

        ax.scatter(
            Z_win[mask, 0], Z_win[mask, 1],
            s=point_size, alpha=alpha, color=color,
            edgecolors="k", linewidths=0.3, label=label, zorder=3
        )

        cx, cy = centroids[tid_int]
        ax.annotate(
            word, (cx, cy),
            textcoords="offset points", xytext=(8, 6),
            fontsize=9, fontweight="bold"
        )

    ax.set_xlabel("PC 1", fontsize=12)
    ax.set_ylabel("PC 2", fontsize=12)
    ax.set_title(
        f"{title_prefix} — PCA of {len(win_ids)} timesteps\n"
        f"Var explained: PC1={pca_win.explained_variance_ratio_[0]:.2%}, "
        f"PC2={pca_win.explained_variance_ratio_[1]:.2%}",
        fontsize=12
    )
    ax.legend(
        bbox_to_anchor=(0.5, -0.1),  # Move to bottom center
        loc="upper center",
        fontsize=9,
        framealpha=0.7,
        title="row",
       # ncol=len(row_labels) if 'row_labels' in locals() else None
    )
    ax.grid(True, alpha=0.15)
    fig.tight_layout()
    plt.show()

    # se ti serve riusare risultati fuori:
    return {
        "win_ids": win_ids,
        "win_emb": win_emb,
        "Z_win": Z_win,
        "centroids": centroids,
        "pca": pca_win,
        "edges": edges,
    }


In [ ]:
ROW_COLORS = ['#e63946', '#457b9d', '#2a9d8f', '#e9c46a']

CONTEXT_LEN = 1800 # in {300, 600, 1200, 1800}

VOCAB_PATH = Path("../data/uncorrelated-words/selected_llama31_layer0.txt")
GRID_PATH  = Path("../data/one_random_walk/grid_16/grid_dataset_structure.txt")
EMB_PATH   = Path(f"../embeddings/one_random_walk/grid_16/reprs_grid_dataset_one_rw_{CONTEXT_LEN}_line000000_layer26.pt")

token_to_id, id_to_token = load_token_id_map(VOCAB_PATH)
GRID, WORD_TO_POS, WORD_TO_TID, ID_TO_WORD, ROW_LABELS = load_grid_structure(GRID_PATH, token_to_id)

all_ids_np, all_emb_np = load_embeddings_pt(EMB_PATH)

print("GRID:")
for row in GRID:
    print(row)

print("\nRow labels:", ROW_LABELS)
print(f"\nEmbeddings shape: {all_emb_np.shape}, ids shape: {all_ids_np.shape}")

token_to_id, id_to_token = load_token_id_map(VOCAB_PATH)

# 2) griglia + dizionari
GRID, WORD_TO_POS, WORD_TO_TID, ID_TO_WORD, ROW_LABELS = load_grid_structure(GRID_PATH, token_to_id)

# 3) embeddings
all_ids_np, all_emb_np = load_embeddings_pt(EMB_PATH)

res = plot_window_pca_grid(
    all_ids_np=all_ids_np,
    all_emb_np=all_emb_np,
    GRID=GRID,
    WORD_TO_POS=WORD_TO_POS,
    WORD_TO_TID=WORD_TO_TID,
    ID_TO_WORD=ID_TO_WORD,
    row_colors=ROW_COLORS,
    row_labels=ROW_LABELS,
    NW=500,
    title_prefix="Single random walk",
)

In [ ]:

CONTEXT_LEN = 1800 # in {300, 600, 1200, 1800}

VOCAB_PATH = Path("../data/paper-tkid-map.txt")
GRID_PATH  = Path("../data/one_random_walk/paper_grid/paper_grid_structure.txt")
EMB_PATH   = Path(f"../embeddings/one_random_walk/paper_grid/reprs_paper_grid_one_rw_{CONTEXT_LEN}_line000000_layer26.pt")

token_to_id, id_to_token = load_token_id_map(VOCAB_PATH)
GRID, WORD_TO_POS, WORD_TO_TID, ID_TO_WORD, ROW_LABELS = load_grid_structure(GRID_PATH, token_to_id)

all_ids_np, all_emb_np = load_embeddings_pt(EMB_PATH)

print("GRID:")
for row in GRID:
    print(row)

print("\nRow labels:", ROW_LABELS)
print(f"\nEmbeddings shape: {all_emb_np.shape}, ids shape: {all_ids_np.shape}")

token_to_id, id_to_token = load_token_id_map(VOCAB_PATH)

# 2) griglia + dizionari
GRID, WORD_TO_POS, WORD_TO_TID, ID_TO_WORD, ROW_LABELS = load_grid_structure(GRID_PATH, token_to_id)

# 3) embeddings
all_ids_np, all_emb_np = load_embeddings_pt(EMB_PATH)

res = plot_window_pca_grid(
    all_ids_np=all_ids_np,
    all_emb_np=all_emb_np,
    GRID=GRID,
    WORD_TO_POS=WORD_TO_POS,
    WORD_TO_TID=WORD_TO_TID,
    ID_TO_WORD=ID_TO_WORD,
    row_colors=ROW_COLORS,
    row_labels=ROW_LABELS,
    NW=500,
    title_prefix="Single random walk",
)

# AAAAAAAAAAA


In [ ]:
# ── General PCA subplot function ──────────────────────────────────────

def plot_pca_on_ax(
    ax,
    all_ids_np,
    all_emb_np,
    ID_TO_WORD,
    token_to_group,    # dict[token] -> group_key (row idx, depth, etc.)
    group_colors,      # dict[group_key] -> color
    group_labels,      # dict[group_key] -> label string
    edges,             # list of (tid1, tid2) — precomputed
    NW=500,
    random_state=0,
    point_size=40,
    alpha=0.7,
    title="",
    annotate=True,
    font_size=7,
):
    """PCA scatter + edges on a given Axes (for subplots)."""
    L = len(all_ids_np)
    W = min(NW, L)
    win_ids = all_ids_np[-W:]
    win_emb = all_emb_np[-W:]

    known_tids = set(ID_TO_WORD.keys())
    keep_mask = np.isin(win_ids, list(known_tids))
    n_discard = int((~keep_mask).sum())
    if n_discard > 0:
        print(f"  [{title}] Discarded {n_discard}/{len(win_ids)} embeddings with unknown token IDs")
    win_ids = win_ids[keep_mask]
    win_emb = win_emb[keep_mask]

    pca = PCA(n_components=2, random_state=random_state)
    Z = pca.fit_transform(win_emb)

    unique_tids = np.unique(win_ids)
    centroids = {int(t): Z[win_ids == t].mean(axis=0) for t in unique_tids}

    avail = set(centroids)
    for t1, t2 in edges:
        if t1 in avail and t2 in avail:
            c1, c2 = centroids[t1], centroids[t2]
            ax.plot([c1[0], c2[0]], [c1[1], c2[1]],
                    color="#cccccc", lw=0.8, zorder=1)

    legend_added = set()
    for tid in unique_tids:
        tid_int = int(tid)
        mask = win_ids == tid
        word = ID_TO_WORD[tid_int]
        grp = token_to_group.get(word)
        color = group_colors.get(grp, "grey")
        label = group_labels.get(grp) if grp not in legend_added else None
        if grp is not None:
            legend_added.add(grp)

        ax.scatter(Z[mask, 0], Z[mask, 1],
                   s=point_size, alpha=alpha, color=color,
                   edgecolors="k", linewidths=0.3, label=label, zorder=3)
        if annotate:
            cx, cy = centroids[tid_int]
            ax.annotate(word, (cx, cy), textcoords="offset points",
                        xytext=(6, 5), fontsize=font_size, fontweight="bold")

    ev = pca.explained_variance_ratio_
    ax.set_xlabel("PC 1", fontsize=9)
    ax.set_ylabel("PC 2", fontsize=9)
    ax.set_title(f"{title}\nVar: {ev[0]:.1%} + {ev[1]:.1%}", fontsize=9)
    ax.legend(fontsize=6, framealpha=0.7, loc="best")
    ax.grid(True, alpha=0.15)


# ── General 3D PCA subplot function ──────────────────────────────────

def plot_pca_3d_on_ax(
    ax,
    all_ids_np,
    all_emb_np,
    ID_TO_WORD,
    token_to_group,
    group_colors,
    group_labels,
    edges,
    NW=500,
    random_state=0,
    point_size=40,
    alpha=0.7,
    title="",
    annotate=True,
    font_size=7,
):
    """PCA 3D scatter + edges on a given 3D Axes (for subplots)."""
    L = len(all_ids_np)
    W = min(NW, L)
    win_ids = all_ids_np[-W:]
    win_emb = all_emb_np[-W:]

    known_tids = set(ID_TO_WORD.keys())
    keep_mask = np.isin(win_ids, list(known_tids))
    n_discard = int((~keep_mask).sum())
    if n_discard > 0:
        print(f"  [{title}] Discarded {n_discard}/{len(win_ids)} embeddings with unknown token IDs")
    win_ids = win_ids[keep_mask]
    win_emb = win_emb[keep_mask]

    pca = PCA(n_components=3, random_state=random_state)
    Z = pca.fit_transform(win_emb)

    unique_tids = np.unique(win_ids)
    centroids = {int(t): Z[win_ids == t].mean(axis=0) for t in unique_tids}

    avail = set(centroids)
    for t1, t2 in edges:
        if t1 in avail and t2 in avail:
            c1, c2 = centroids[t1], centroids[t2]
            ax.plot([c1[0], c2[0]], [c1[1], c2[1]], [c1[2], c2[2]],
                    color="#cccccc", lw=0.8, zorder=1)

    legend_added = set()
    for tid in unique_tids:
        tid_int = int(tid)
        mask = win_ids == tid
        word = ID_TO_WORD[tid_int]
        grp = token_to_group.get(word)
        color = group_colors.get(grp, "grey")
        label = group_labels.get(grp) if grp not in legend_added else None
        if grp is not None:
            legend_added.add(grp)

        ax.scatter(Z[mask, 0], Z[mask, 1], Z[mask, 2],
                   s=point_size, alpha=alpha, color=color,
                   edgecolors="k", linewidths=0.3, label=label, zorder=3)
        if annotate:
            cx, cy, cz = centroids[tid_int]
            ax.text(cx, cy, cz, f"  {word}",
                    fontsize=font_size, fontweight="bold", zorder=4)

    ev = pca.explained_variance_ratio_
    ax.set_xlabel("PC 1", fontsize=8)
    ax.set_ylabel("PC 2", fontsize=8)
    ax.set_zlabel("PC 3", fontsize=8)
    ax.set_title(
        f"{title}\nVar: {ev[0]:.1%} + {ev[1]:.1%} + {ev[2]:.1%}",
        fontsize=9,
    )
    ax.legend(fontsize=6, framealpha=0.7, loc="best")


# ── Torus edges (grid with wrap-around) ──────────────────────────────

def torus_edges_from_grid(GRID, WORD_TO_TID):
    """Like grid_edges_from_grid but top↔bottom and left↔right wrap."""
    n_rows, n_cols = len(GRID), len(GRID[0])
    edges = set()
    for r in range(n_rows):
        for c in range(n_cols):
            t1 = WORD_TO_TID[GRID[r][c]]
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                rr, cc = (r + dr) % n_rows, (c + dc) % n_cols
                t2 = WORD_TO_TID[GRID[rr][cc]]
                if t1 != t2:
                    edges.add((min(t1, t2), max(t1, t2)))
    return list(edges)


# ── Tree structure parser ─────────────────────────────────────────────

def load_tree_structure(path, token_to_id, encoding="utf-8"):
    """
    Parse ASCII tree file (├── / └── format).
    Returns: tokens, token_to_depth, edges (tid pairs),
             WORD_TO_TID, ID_TO_WORD, depth_labels
    """
    lines = path.read_text(encoding=encoding).strip().splitlines()
    tokens = []
    token_to_depth = {}
    parent_at_depth = {}
    edges_words = []

    for line in lines:
        stripped = line.rstrip()
        if not stripped:
            continue

        pos = None
        for j, ch in enumerate(stripped):
            if ch in ('├', '└'):
                pos = j
                break

        if pos is None:
            token = stripped.strip()
            depth = 0
        else:
            depth = pos // 4 + 1
            token = stripped[pos:].lstrip('├└─ ').strip()

        tokens.append(token)
        token_to_depth[token] = depth
        parent_at_depth[depth] = token
        if depth > 0 and (depth - 1) in parent_at_depth:
            edges_words.append((parent_at_depth[depth - 1], token))

    missing = [t for t in tokens if t not in token_to_id]
    if missing:
        raise KeyError(f"Missing tokens: {missing[:8]}")

    WORD_TO_TID = {t: token_to_id[t] for t in tokens}
    ID_TO_WORD  = {tid: t for t, tid in WORD_TO_TID.items()}

    edges = [(min(WORD_TO_TID[p], WORD_TO_TID[c]),
              max(WORD_TO_TID[p], WORD_TO_TID[c]))
             for p, c in edges_words]

    max_d = max(token_to_depth.values())
    depth_labels = {d: f"Depth {d}" for d in range(max_d + 1)}

    return tokens, token_to_depth, edges, WORD_TO_TID, ID_TO_WORD, depth_labels


# ── Cluster-tree structure parser ─────────────────────────────────────

def load_tree_cluster_structure(path, token_to_id, encoding="utf-8"):
    """
    Parse cluster tree where each node = (tok1, tok2, tok3).
    Returns: clusters, token_to_depth,
             inter_edges (parent↔child cluster), intra_edges (within cluster),
             WORD_TO_TID, ID_TO_WORD, depth_labels
    """
    lines = path.read_text(encoding=encoding).strip().splitlines()
    clusters = []
    cluster_depths = []
    parent_cluster_at_depth = {}
    token_to_depth = {}
    tree_links = []

    for line in lines:
        stripped = line.rstrip()
        if not stripped:
            continue
        try:
            s, e = stripped.index('('), stripped.index(')')
        except ValueError:
            continue

        cluster_tokens = [t.strip() for t in stripped[s + 1:e].split(',')]

        pos = None
        for j, ch in enumerate(stripped):
            if ch in ('├', '└'):
                pos = j
                break
        depth = 0 if pos is None else pos // 4 + 1

        idx = len(clusters)
        clusters.append(cluster_tokens)
        cluster_depths.append(depth)
        for t in cluster_tokens:
            token_to_depth[t] = depth
        parent_cluster_at_depth[depth] = idx
        if depth > 0 and (depth - 1) in parent_cluster_at_depth:
            tree_links.append((parent_cluster_at_depth[depth - 1], idx))

    all_tokens = [t for cl in clusters for t in cl]
    missing = [t for t in all_tokens if t not in token_to_id]
    if missing:
        raise KeyError(f"Missing tokens: {missing[:8]}")

    WORD_TO_TID = {t: token_to_id[t] for t in all_tokens}
    ID_TO_WORD  = {tid: t for t, tid in WORD_TO_TID.items()}

    inter_edges = []
    for pi, ci in tree_links:
        for pt in clusters[pi]:
            for ct in clusters[ci]:
                t1, t2 = WORD_TO_TID[pt], WORD_TO_TID[ct]
                inter_edges.append((min(t1, t2), max(t1, t2)))

    intra_edges = []
    for cl in clusters:
        for a in range(len(cl)):
            for b in range(a + 1, len(cl)):
                t1, t2 = WORD_TO_TID[cl[a]], WORD_TO_TID[cl[b]]
                intra_edges.append((min(t1, t2), max(t1, t2)))

    max_d = max(token_to_depth.values())
    depth_labels = {d: f"Depth {d}" for d in range(max_d + 1)}

    return (clusters, token_to_depth, inter_edges, intra_edges,
            WORD_TO_TID, ID_TO_WORD, depth_labels)


# ── Dendrogram from embedding centroids ───────────────────────────────

def plot_dendrogram_on_ax(
    ax,
    all_ids_np,
    all_emb_np,
    ID_TO_WORD,
    token_to_depth,
    depth_colors,
    NW=500,
    title="",
    leaf_font_size=7,
    method="ward",
):
    """
    Compute per-token centroids, run hierarchical clustering (Ward),
    and draw a horizontal dendrogram. Leaves colored by true tree depth.
    """
    L = len(all_ids_np)
    W = min(NW, L)
    win_ids = all_ids_np[-W:]
    win_emb = all_emb_np[-W:]

    known_tids = set(ID_TO_WORD.keys())
    keep_mask = np.isin(win_ids, list(known_tids))
    n_discard = int((~keep_mask).sum())
    if n_discard > 0:
        print(f"  [{title}] Discarded {n_discard}/{len(win_ids)} embeddings with unknown token IDs")
    win_ids = win_ids[keep_mask]
    win_emb = win_emb[keep_mask]

    unique_tids = np.unique(win_ids)
    labels = []
    centroids = []
    leaf_colors = []
    for tid in unique_tids:
        tid_int = int(tid)
        word = ID_TO_WORD[tid_int]
        labels.append(word)
        centroids.append(win_emb[win_ids == tid].mean(axis=0))
        depth = token_to_depth.get(word, 0)
        leaf_colors.append(depth_colors.get(depth, "grey"))

    centroids = np.stack(centroids)
    Z = linkage(centroids, method=method)

    leaf_color_map = {lab: col for lab, col in zip(labels, leaf_colors)}

    dendro = dendrogram(
        Z,
        labels=labels,
        ax=ax,
        orientation="left",
        leaf_font_size=leaf_font_size,
        link_color_func=lambda k: "#888888",
    )

    y_labels = ax.get_yticklabels()
    for lbl in y_labels:
        txt = lbl.get_text()
        lbl.set_color(leaf_color_map.get(txt, "black"))
        lbl.set_fontweight("bold")

    ax.set_title(title, fontsize=9)
    ax.set_xlabel("Ward distance", fontsize=9)
    ax.tick_params(axis="y", labelsize=leaf_font_size)


# ── Heatmap of pairwise distances between embedding centroids ─────────






# ── UMAP of embedding centroids ──────────────────────────────────────

def plot_umap_on_ax(
    ax,
    all_ids_np,
    all_emb_np,
    ID_TO_WORD,
    token_to_group,
    group_colors,
    group_labels,
    edges,
    NW=500,
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=0,
    point_size=40,
    alpha=0.7,
    title="",
    annotate=True,
    font_size=7,
):
    """
    Compute per-token centroids, project with UMAP to 2D or 3D,
    scatter-plot colored by group (e.g. tree depth), draw edges.
    Pass n_components=3 and a 3D axes (projection='3d') for 3D plots.
    """
    L = len(all_ids_np)
    W = min(NW, L)
    win_ids = all_ids_np[-W:]
    win_emb = all_emb_np[-W:]

    known_tids = set(ID_TO_WORD.keys())
    keep_mask = np.isin(win_ids, list(known_tids))
    n_discard = int((~keep_mask).sum())
    if n_discard > 0:
        print(f"  [{title}] Discarded {n_discard}/{len(win_ids)} embeddings with unknown token IDs")
    win_ids = win_ids[keep_mask]
    win_emb = win_emb[keep_mask]

    unique_tids = np.unique(win_ids)
    labels_list = []
    centroids_list = []
    for tid in unique_tids:
        tid_int = int(tid)
        word = ID_TO_WORD[tid_int]
        labels_list.append(word)
        centroids_list.append(win_emb[win_ids == tid].mean(axis=0))

    centroids_arr = np.stack(centroids_list)

    n_nb = min(n_neighbors, len(centroids_arr) - 1)
    reducer = umap.UMAP(n_components=n_components, n_neighbors=n_nb,
                        min_dist=min_dist, random_state=random_state)
    Z = reducer.fit_transform(centroids_arr)

    is_3d = n_components >= 3

    tid_to_idx = {int(tid): i for i, tid in enumerate(unique_tids)}
    avail = set(tid_to_idx.keys())
    for t1, t2 in edges:
        if t1 in avail and t2 in avail:
            i1, i2 = tid_to_idx[t1], tid_to_idx[t2]
            if is_3d:
                ax.plot([Z[i1, 0], Z[i2, 0]], [Z[i1, 1], Z[i2, 1]],
                        [Z[i1, 2], Z[i2, 2]], color="#cccccc", lw=0.8, zorder=1)
            else:
                ax.plot([Z[i1, 0], Z[i2, 0]], [Z[i1, 1], Z[i2, 1]],
                        color="#cccccc", lw=0.8, zorder=1)

    legend_added = set()
    for i, word in enumerate(labels_list):
        grp = token_to_group.get(word)
        color = group_colors.get(grp, "grey")
        label = group_labels.get(grp) if grp not in legend_added else None
        if grp is not None:
            legend_added.add(grp)

        if is_3d:
            ax.scatter(Z[i, 0], Z[i, 1], Z[i, 2],
                       s=point_size, alpha=alpha, color=color,
                       edgecolors="k", linewidths=0.3, label=label, zorder=3)
            if annotate:
                ax.text(Z[i, 0], Z[i, 1], Z[i, 2], f"  {word}",
                        fontsize=font_size, fontweight="bold", zorder=4)
        else:
            ax.scatter(Z[i, 0], Z[i, 1],
                       s=point_size, alpha=alpha, color=color,
                       edgecolors="k", linewidths=0.3, label=label, zorder=3)
            if annotate:
                ax.annotate(word, (Z[i, 0], Z[i, 1]), textcoords="offset points",
                            xytext=(6, 5), fontsize=font_size, fontweight="bold")

    ax.set_xlabel("UMAP 1", fontsize=9)
    ax.set_ylabel("UMAP 2", fontsize=9)
    if is_3d:
        ax.set_zlabel("UMAP 3", fontsize=9)
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=6, framealpha=0.7, loc="best")
    if not is_3d:
        ax.grid(True, alpha=0.15)

# Torus


In [ ]:
VOCAB_PATH  = Path("../data/uncorrelated-words/selected_llama31_layer0.txt")
STRUCT_PATH = Path("../data/one_random_walk/torus_16/torus_dataset_structure.txt")
EMB_DIR     = Path("../embeddings/one_random_walk/torus_16")

token_to_id, _ = load_token_id_map(VOCAB_PATH)
GRID, WORD_TO_POS, WORD_TO_TID, ID_TO_WORD, ROW_LABELS = \
    load_grid_structure(STRUCT_PATH, token_to_id)

all_edges = torus_edges_from_grid(GRID, WORD_TO_TID)

ROW_COLORS_MAP = {0: '#e63946', 1: '#457b9d', 2: '#2a9d8f', 3: '#e9c46a'}
ROW_LABELS_MAP = {r: f"Row {r}: " + " ".join(GRID[r]) for r in range(4)}
token_to_group = {w: r for r, row in enumerate(GRID) for c, w in enumerate(row)}

CTX_LENS = [300, 600, 1200, 1800]
fig = plt.figure(figsize=(18, 16))

for i, ctx in enumerate(CTX_LENS, start=1):
    ax = fig.add_subplot(2, 2, i, projection='3d')
    pt_path = EMB_DIR / f"reprs_torus_dataset_one_rw_{ctx}_line000000_layer26.pt"
    ids_np, emb_np = load_embeddings_pt(pt_path)
    plot_pca_3d_on_ax(
        ax, ids_np, emb_np, ID_TO_WORD,
        token_to_group, ROW_COLORS_MAP, ROW_LABELS_MAP,
        all_edges, title=f"Torus — ctx {ctx}",
    )

fig.suptitle("Torus 4×4 — 3D PCA of single random walk embeddings",
             fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
CTX_LENS = [300, 600, 1200, 1800]
fig = plt.figure(figsize=(18, 16))

for i, ctx in enumerate(CTX_LENS, start=1):
    ax = fig.add_subplot(2, 2, i, projection='3d')
    pt_path = EMB_DIR / f"reprs_torus_dataset_one_rw_{ctx}_line000000_layer26.pt"
    ids_np, emb_np = load_embeddings_pt(pt_path)
    plot_umap_on_ax(
        ax, ids_np, emb_np, ID_TO_WORD,
        token_to_group, ROW_COLORS_MAP, ROW_LABELS_MAP,
        all_edges, title=f"Torus UMAP — ctx {ctx}",
        n_components=3
    )

fig.suptitle("Torus 4×4 — UMAP of embedding centroids",
             fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

# Tree

In [ ]:
VOCAB_PATH  = Path("../data/uncorrelated-words/selected_llama31_layer0.txt")
STRUCT_PATH = Path("../data/one_random_walk/tree_4_levels/bin_tree_structure.txt")
EMB_DIR     = Path("../embeddings/one_random_walk/tree_4_levels")

token_to_id, _ = load_token_id_map(VOCAB_PATH)
tokens, token_to_depth, tree_edges, WORD_TO_TID, ID_TO_WORD, depth_labels = \
    load_tree_structure(STRUCT_PATH, token_to_id)

DEPTH_COLORS = {0: '#e63946', 1: '#457b9d', 2: '#2a9d8f', 3: '#e9c46a', 4: '#f4a261'}

CTX_LENS = [300, 600, 1200, 1800]
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for ax, ctx in zip(axes.flat, CTX_LENS):
    pt_path = EMB_DIR / f"reprs_bin_tree_one_rw_{ctx}_line000000_layer26.pt"
    ids_np, emb_np = load_embeddings_pt(pt_path)
    plot_pca_on_ax(
        ax, ids_np, emb_np, ID_TO_WORD,
        token_to_depth, DEPTH_COLORS, depth_labels,
        tree_edges, title=f"Binary tree — ctx {ctx}",
        font_size=6,
    )

fig.suptitle("Binary Tree (4 levels) — PCA of single random walk embeddings",
             fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
DEPTH_COLORS = {0: '#e63946', 1: '#457b9d', 2: '#2a9d8f', 3: '#e9c46a', 4: '#f4a261'}

CTX_LENS = [300, 600, 1200, 1800]
fig, axes = plt.subplots(2, 2, figsize=(18, 12))

for ax, ctx in zip(axes.flat, CTX_LENS):
    pt_path = EMB_DIR / f"reprs_bin_tree_one_rw_{ctx}_line000000_layer26.pt"
    ids_np, emb_np = load_embeddings_pt(pt_path)
    plot_dendrogram_on_ax(
        ax, ids_np, emb_np, ID_TO_WORD,
        token_to_depth, DEPTH_COLORS,
        title=f"Binary tree — ctx {ctx}",
        leaf_font_size=6,
    )

fig.suptitle("Binary Tree (4 levels) — Dendrogram of embedding centroids (Ward)",
             fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
DEPTH_COLORS = {0: '#e63946', 1: '#457b9d', 2: '#2a9d8f', 3: '#e9c46a', 4: '#f4a261'}

CTX_LENS = [300, 600, 1200, 1800]
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for ax, ctx in zip(axes.flat, CTX_LENS):
    pt_path = EMB_DIR / f"reprs_bin_tree_one_rw_{ctx}_line000000_layer26.pt"
    ids_np, emb_np = load_embeddings_pt(pt_path)
    plot_umap_on_ax(
        ax, ids_np, emb_np, ID_TO_WORD,
        token_to_depth, DEPTH_COLORS, depth_labels,
        tree_edges, title=f"Binary tree UMAP — ctx {ctx}",
        font_size=6,
    )

fig.suptitle("Binary Tree (4 levels) — UMAP of embedding centroids",
             fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
DEPTH_COLORS = {0: '#e63946', 1: '#457b9d', 2: '#2a9d8f', 3: '#e9c46a', 4: '#f4a261'}

CTX_LENS = [300, 600, 1200, 1800]
fig = plt.figure(figsize=(18, 16))

for i, ctx in enumerate(CTX_LENS, start=1):
    ax = fig.add_subplot(2, 2, i, projection='3d')
    pt_path = EMB_DIR / f"reprs_bin_tree_one_rw_{ctx}_line000000_layer26.pt"
    ids_np, emb_np = load_embeddings_pt(pt_path)
    plot_umap_on_ax(
        ax, ids_np, emb_np, ID_TO_WORD,
        token_to_depth, DEPTH_COLORS, depth_labels,
        tree_edges, title=f"Binary tree UMAP — ctx {ctx}",
        font_size=6, n_components=3
    )

fig.suptitle("Binary Tree (4 levels) — UMAP of embedding centroids",
             fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

# Cluster Tree


In [ ]:
VOCAB_PATH  = Path("../data/uncorrelated-words/selected_llama31_layer0.txt")
STRUCT_PATH = Path("../data/one_random_walk/tree_clusters_3_levels/bin_tree_cluster_structure.txt")
EMB_DIR     = Path("../embeddings/one_random_walk/tree_clusters_3_levels")

token_to_id, _ = load_token_id_map(VOCAB_PATH)
clusters, token_to_depth, inter_edges, intra_edges, WORD_TO_TID, ID_TO_WORD, depth_labels = \
    load_tree_cluster_structure(STRUCT_PATH, token_to_id)

DEPTH_COLORS = {0: '#e63946', 1: '#457b9d', 2: '#2a9d8f', 3: '#e9c46a'}
all_edges = inter_edges + intra_edges

CTX_LENS = [300, 600, 1200, 1800]
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

for ax, ctx in zip(axes.flat, CTX_LENS):
    pt_path = EMB_DIR / f"reprs_bin_tree_cluster_one_rw_{ctx}_line000000_layer26.pt"
    ids_np, emb_np = load_embeddings_pt(pt_path)
    plot_pca_on_ax(
        ax, ids_np, emb_np, ID_TO_WORD,
        token_to_depth, DEPTH_COLORS, depth_labels,
        all_edges, title=f"Cluster tree — ctx {ctx}",
        font_size=5, point_size=30,
    )

fig.suptitle("Cluster Binary Tree (3 levels, 3 tokens/node) — PCA embeddings",
             fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
DEPTH_COLORS = {0: '#e63946', 1: '#457b9d', 2: '#2a9d8f', 3: '#e9c46a'}

CTX_LENS = [300, 600, 1200, 1800]
fig, axes = plt.subplots(2, 2, figsize=(20, 12))

for ax, ctx in zip(axes.flat, CTX_LENS):
    pt_path = EMB_DIR / f"reprs_bin_tree_cluster_one_rw_{ctx}_line000000_layer26.pt"
    ids_np, emb_np = load_embeddings_pt(pt_path)
    plot_dendrogram_on_ax(
        ax, ids_np, emb_np, ID_TO_WORD,
        token_to_depth, DEPTH_COLORS,
        title=f"Cluster tree — ctx {ctx}",
        leaf_font_size=5,
    )

fig.suptitle("Cluster Binary Tree (3 levels, 3 tokens/node) — Dendrogram of embedding centroids (Ward)",
             fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
DEPTH_COLORS = {0: '#e63946', 1: '#457b9d', 2: '#2a9d8f', 3: '#e9c46a'}

CTX_LENS = [300, 600, 1200, 1800]
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

for ax, ctx in zip(axes.flat, CTX_LENS):
    pt_path = EMB_DIR / f"reprs_bin_tree_cluster_one_rw_{ctx}_line000000_layer26.pt"
    ids_np, emb_np = load_embeddings_pt(pt_path)
    plot_umap_on_ax(
        ax, ids_np, emb_np, ID_TO_WORD,
        token_to_depth, DEPTH_COLORS, depth_labels,
        all_edges, title=f"Cluster tree UMAP — ctx {ctx}",
        font_size=5, point_size=30
    )

fig.suptitle("Cluster Binary Tree (3 levels, 3 tokens/node) — UMAP of embedding centroids",
             fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
DEPTH_COLORS = {0: '#e63946', 1: '#457b9d', 2: '#2a9d8f', 3: '#e9c46a'}

CTX_LENS = [300, 600, 1200, 1800]
fig = plt.figure(figsize=(18, 16))
for i, ctx in enumerate(CTX_LENS, start=1):
    ax = fig.add_subplot(2, 2, i, projection='3d')
    pt_path = EMB_DIR / f"reprs_bin_tree_cluster_one_rw_{ctx}_line000000_layer26.pt"
    ids_np, emb_np = load_embeddings_pt(pt_path)
    plot_umap_on_ax(
        ax, ids_np, emb_np, ID_TO_WORD,
        token_to_depth, DEPTH_COLORS, depth_labels,
        all_edges, title=f"Cluster tree UMAP — ctx {ctx}",
        font_size=5, point_size=30,n_components=3
    )

fig.suptitle("Cluster Binary Tree (3 levels, 3 tokens/node) — UMAP of embedding centroids",
             fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

# Parametrized Grid

In [ ]:
VOCAB_PATH  = Path("../data/uncorrelated-words/selected_llama31_layer0.txt")
STRUCT_PATH = Path("../data/one_random_walk/parametrized_grid_16/grid_dataset_structure.txt")
EMB_DIR     = Path("../embeddings/one_random_walk/parametrized_grid_16")

token_to_id, _ = load_token_id_map(VOCAB_PATH)
GRID, WORD_TO_POS, WORD_TO_TID, ID_TO_WORD, ROW_LABELS = \
    load_grid_structure(STRUCT_PATH, token_to_id)

all_edges = grid_edges_from_grid(GRID, WORD_TO_TID)

ROW_COLORS_MAP = {0: '#e63946', 1: '#457b9d', 2: '#2a9d8f', 3: '#e9c46a'}
ROW_LABELS_MAP = {r: f"Row {r}: " + " ".join(GRID[r]) for r in range(4)}
token_to_group = {w: r for r, row in enumerate(GRID) for c, w in enumerate(row)}

CTX_LENS = [300, 600, 1200, 1800]
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for ax, ctx in zip(axes.flat, CTX_LENS):
    pt_path = EMB_DIR / f"reprs_grid_dataset_one_rw_{ctx}_line000000_layer26.pt"
    ids_np, emb_np = load_embeddings_pt(pt_path)
    plot_pca_on_ax(
        ax, ids_np, emb_np, ID_TO_WORD,
        token_to_group, ROW_COLORS_MAP, ROW_LABELS_MAP,
        all_edges, title=f"Parametrized grid — ctx {ctx}",
    )

fig.suptitle("Parametrized Grid 4×4 — PCA of single random walk embeddings",
             fontsize=14, y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
CTX_LENS = [300, 600, 1200, 1800]
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for ax, ctx in zip(axes.flat, CTX_LENS):
    pt_path = EMB_DIR / f"reprs_grid_dataset_one_rw_{ctx}_line000000_layer26.pt"
    ids_np, emb_np = load_embeddings_pt(pt_path)
    plot_umap_on_ax(
        ax, ids_np, emb_np, ID_TO_WORD,
        token_to_group, ROW_COLORS_MAP, ROW_LABELS_MAP,
        all_edges, title=f"Parametrized grid UMAP — ctx {ctx}",
    )

fig.suptitle("Parametrized Grid 4×4 — UMAP of embedding centroids",
             fontsize=14, y=1.01)
fig.tight_layout()
plt.show()